In [1]:
import os
import shutil
import argparse

import numpy as np
import pandas as pd
import polars as pl

import pydicom
import cv2
from scipy import ndimage

import torch
import torchvision
import torchvision.models as models


print(f'torch version: {torch.__version__}')
print(f'torchvision version: {torchvision.__version__}')

torch version: 2.7.1+cu126
torchvision version: 0.22.1+cu126


In [2]:
ID_COL = 'SeriesInstanceUID'

LABEL_COLS = [
    'Left Infraclinoid Internal Carotid Artery',
    'Right Infraclinoid Internal Carotid Artery',
    'Left Supraclinoid Internal Carotid Artery',
    'Right Supraclinoid Internal Carotid Artery',
    'Left Middle Cerebral Artery',
    'Right Middle Cerebral Artery',
    'Anterior Communicating Artery',
    'Left Anterior Cerebral Artery',
    'Right Anterior Cerebral Artery',
    'Left Posterior Communicating Artery',
    'Right Posterior Communicating Artery',
    'Basilar Tip',
    'Other Posterior Circulation',
    'Aneurysm Present',
]

NUM_CLASSES = len(LABEL_COLS)

In [3]:
def parse_args(arg_params: list[str]|None=None) -> argparse.Namespace:
    parser = argparse.ArgumentParser()
    
    # Image size
    parser.add_argument('-size', '--img_size', default=416, type=int, help='input image size')
    
    # Model
    parser.add_argument('-m', '--model', default=torchvision.models.resnet18.__name__.lower(), type=str, help='select model')
    parser.add_argument('--num_fold', default=1, type=int, help='number of K-fold')
    parser.add_argument('--input_channels', default=32, type=int, help='number of model input channels')
    parser.add_argument('--model_weight_path', default='./checkpoints/best_checkpoint_resnet18_20251005_021901.pth', type=str, help='Trained state_dict file path to open')

    args = parser.parse_args(arg_params)

    return args

In [4]:
args = parse_args([
    "--model", "resnet18",
    "--num_fold", "5",
    "--model_weight_path", "./checkpoints/resnet18_20251012_150358/",
    ])
print("Setting Arguments.. : ", args)

Setting Arguments.. :  Namespace(img_size=416, model='resnet18', num_fold=5, input_channels=32, model_weight_path='./checkpoints/resnet18_20251012_150358/')


In [5]:
device = torch.device(torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu')
print(f'device: {device}')

device: cuda


In [6]:
def build_resnet(resnet_name: str, in_channels: int, num_classes: int) -> torch.nn.Module|None:

    resnet = None
    
    if models.resnet18.__name__.lower() == resnet_name:
        resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
    elif models.resnet34.__name__.lower() == resnet_name:
        resnet = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
    elif models.resnet50.__name__.lower() == resnet_name:
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
    elif models.resnet101.__name__.lower() == resnet_name:
        resnet = models.resnet101(weights=models.ResNet101_Weights.DEFAULT)
    elif models.resnet152.__name__.lower() == resnet_name:
        resnet = models.resnet152(weights=models.ResNet152_Weights.DEFAULT)
    
    if resnet is not None:
        conv1 = resnet.conv1
        resnet.conv1 = torch.nn.Conv2d(in_channels, conv1.out_channels
                                       , conv1.kernel_size[0], conv1.stride[0], conv1.padding[0], bias=False)
        torch.nn.init.kaiming_uniform_(resnet.conv1.weight)

        fc_in_features = resnet.fc.in_features
        resnet.fc = torch.nn.Linear(fc_in_features, num_classes)
        
        gain = torch.nn.init.calculate_gain('linear')
        torch.nn.init.xavier_uniform_(resnet.fc.weight, gain)
        resnet.fc.bias.data.fill_(0)

    return resnet

def build_backbone(backbone_name: str, in_channels: int, num_classes: int) -> torch.nn.Module|None:

    backbone = None
    
    if backbone_name.startswith(models.ResNet.__name__.lower()):
        # ResNet base
        backbone = build_resnet(backbone_name, in_channels, num_classes)

    return backbone


class DiseaseDetector(torch.nn.Module):
    def __init__(self, backbone_name: str, in_channels: int, num_classes: int):
        super(DiseaseDetector, self).__init__()

        # ResNet base
        self.classifier = build_backbone(backbone_name, in_channels, num_classes)

        if self.classifier is None:
            raise ModuleNotFoundError(name=backbone_name)
    
    def forward(self, image: torch.Tensor) -> torch.Tensor:
        return self.classifier(image)
    
    @torch.no_grad()
    def inference(self, image: torch.Tensor) -> torch.Tensor:
        if 3 == len(image.shape):
            image = image.unsqueeze(0)
        
        return self.forward(image).sigmoid()


def build_model(args: argparse.Namespace, num_classes: int) -> list[torch.nn.Module]:
    return [DiseaseDetector(args.model, args.input_channels, num_classes) for _ in range(args.num_fold)]

In [7]:
model_loaders = build_model(args, NUM_CLASSES)
models = []

for paht_index, path_value in enumerate(os.walk(args.model_weight_path)):
    root, folders, files = path_value
    for file_index, file in enumerate(files):
         if file.endswith('.pth'):
            model_loaders[file_index].load_state_dict(torch.load(os.path.join(args.model_weight_path, file)))
            models.append(model_loaders[file_index].to(device))

In [8]:
class DICOMPreprocessor:
    def __init__(self, target_shape: tuple[int, int, int] = (32, 352, 352)):
        self.target_depth, self.target_height, self.target_width = target_shape

    def __call__(self, series_path: str) -> tuple[str, str, np.ndarray]:
        series_uid = os.path.basename(series_path)
        dicom_datasets = []

        for paht_index, path_value in enumerate(os.walk(series_path)):
            root, folders, files = path_value
            for file_index, file in enumerate(files):
                if file.endswith('.dcm'):
                    dataset = pydicom.dcmread(os.path.join(root, file), force=True)
                    dicom_datasets.append(dataset)
        
        if 0 >= len(dicom_datasets):
            print(f'Invalid dicom series path, series_path: {series_path}')
            return series_uid, 'None', np.ndarray([])
        
        first_image = dicom_datasets[0]
        modality = getattr(first_image, 'Modality')
        if None == modality:
            print(f'invalid modality, series_uid: {series_uid}, file: {first_image}')
        
        if len(dicom_datasets) == 1 and first_image.pixel_array.ndim == 3:
            slope = getattr(first_image, 'RescaleSlope', 1)
            intercept = getattr(first_image, 'RescaleIntercept', 0)
            sorted_slices = [(slope, intercept, first_image.pixel_array[depth_index]) for depth_index in range(first_image.pixel_array.shape[0])]
        else:
            slice_info = self.extract_slice_info(dicom_datasets)
            sorted_slices = []
            for sorted_slice in sorted(slice_info, key=lambda x: x['z_position']):
                ds = sorted_slice['dataset']
                slope = getattr(ds, 'RescaleSlope', 1)
                intercept = getattr(ds, 'RescaleIntercept', 0)
                sorted_slices.append((slope, intercept, ds.pixel_array))

        image_arrays = []

        for slice_data in sorted_slices:
            slope, intercept, pixel_array = slice_data
            
            # Get pixel array
            image = pixel_array.astype(np.float32)
            
            # Apply RescaleSlope and RescaleIntercept
            if slope != 1 or intercept != 0:
                image = image * float(slope) + float(intercept)

            normalized_image = self.apply_windowing_or_normalize(modality, image)
            resized_img = cv2.resize(normalized_image, (self.target_width, self.target_height))            
            image_arrays.append(resized_img)
        
        image_arrays = np.stack(image_arrays, axis=0)

        image_arrays = self.resize_volume_3d(image_arrays)
        
        return series_uid, modality, image_arrays

    def extract_slice_info(self, datasets: list[pydicom.Dataset]) -> list[dict]:
        """
        Extract position information for each slice
        """
        slice_info = []
        
        for i, ds in enumerate(datasets):
            info = {
                'dataset': ds,
                'index': i,
                'instance_number': getattr(ds, 'InstanceNumber', i),
            }
            
            # Get z-coordinate from ImagePositionPatient
            try:
                position = getattr(ds, 'ImagePositionPatient', None)
                if position is not None and len(position) >= 3:
                    info['z_position'] = float(position[2])
                else:
                    # Fallback: use InstanceNumber
                    info['z_position'] = float(info['instance_number'])
                    #print("ImagePositionPatient not found, using InstanceNumber")
            except Exception as e:
                info['z_position'] = float(i)
                #print(f"Failed to extract position info: {e}")
            
            slice_info.append(info)
        
        return slice_info

    def apply_windowing_or_normalize(self, modality: str, img: np.ndarray) -> np.ndarray:
        """
        Apply windowing or statistical normalization
        """
        if modality == 'CT':
            # # Windowing processing (for CT/CTA)
            # img_min = center - width / 2
            # img_max = center + width / 2
            
            # windowed = np.clip(img, img_min, img_max)
            # windowed = (windowed - img_min) / (img_max - img_min + 1e-7)
            # result = (windowed * 255).astype(np.uint8)
            
            # #print(f"Applied windowing: [{img_min:.1f}, {img_max:.1f}] → [0, 255]")
            # return result
            
            # Statistical normalization (for CT as well)
            # Normalize using 1-99 percentiles
            p1, p99 = np.percentile(img, [1, 99])
            p1, p99 = 0, 500
            
            if p99 > p1:
                normalized = np.clip(img, p1, p99)
                normalized = (normalized - p1) / (p99 - p1)
                result = (normalized * 255).astype(np.uint8)
                
                #print(f"Applied statistical normalization: [{p1:.1f}, {p99:.1f}] → [0, 255]")
                return result
            else:
                # Fallback: min-max normalization
                img_min, img_max = img.min(), img.max()
                if img_max > img_min:
                    normalized = (img - img_min) / (img_max - img_min)
                    result = (normalized * 255).astype(np.uint8)
                    #print(f"Applied min-max normalization: [{img_min:.1f}, {img_max:.1f}] → [0, 255]")
                    return result
                else:
                    # If image has no variation
                    #print("Image has no variation, returning zeros")
                    return np.zeros_like(img, dtype=np.uint8)
        
        else:
            # Statistical normalization (for MR)
            # Normalize using 1-99 percentiles
            p1, p99 = np.percentile(img, [1, 99])
            
            if p99 > p1:
                normalized = np.clip(img, p1, p99)
                normalized = (normalized - p1) / (p99 - p1)
                result = (normalized * 255).astype(np.uint8)
                
                #print(f"Applied statistical normalization: [{p1:.1f}, {p99:.1f}] → [0, 255]")
                return result
            else:
                # Fallback: min-max normalization
                img_min, img_max = img.min(), img.max()
                if img_max > img_min:
                    normalized = (img - img_min) / (img_max - img_min)
                    result = (normalized * 255).astype(np.uint8)
                    #print(f"Applied min-max normalization: [{img_min:.1f}, {img_max:.1f}] → [0, 255]")
                    return result
                else:
                    # If image has no variation
                    #print("Image has no variation, returning zeros")
                    return np.zeros_like(img, dtype=np.uint8)

    def resize_volume_3d(self, volume: np.ndarray) -> np.ndarray:
        """
        Resize 3D volume to target size
        """
        current_shape = volume.shape
        target_shape = (self.target_depth, self.target_height, self.target_width)
        
        if current_shape == target_shape:
            return volume
        
        #print(f"Resizing volume from {current_shape} to {target_shape}")
        
        # 3D resizing using scipy.ndimage
        zoom_factors = [
            target_shape[i] / current_shape[i] for i in range(3)
        ]
        
        # Resize with linear interpolation
        resized_volume = ndimage.zoom(volume, zoom_factors, order=1, mode='nearest')
        
        # Clip to exact size just in case
        resized_volume = resized_volume[:self.target_depth, :self.target_height, :self.target_width]
        
        # Padding if necessary
        pad_width = [
            (0, max(0, self.target_depth - resized_volume.shape[0])),
            (0, max(0, self.target_height - resized_volume.shape[1])),
            (0, max(0, self.target_width - resized_volume.shape[2]))
        ]
        
        if any(pw[1] > 0 for pw in pad_width):
            resized_volume = np.pad(resized_volume, pad_width, mode='edge')
        
        #print(f"Final volume shape: {resized_volume.shape}")
        return resized_volume.astype(np.uint8)

In [9]:
preprocessor = DICOMPreprocessor((args.input_channels, args.img_size, args.img_size))

In [10]:
def predict(series_path: str) -> pl.DataFrame | pd.DataFrame:

    series_uid = os.path.basename(series_path)

    try:
        # Extract series ID
        series_uid, modality, image_arrays = preprocessor(series_path)

        image_arrays = torch.from_numpy(image_arrays).contiguous().float() / 255.
        image_arrays = image_arrays.to(device)

        predicts = []

        for index, model in enumerate(models):
            predict = model.inference(image_arrays)
            predict = predict.squeeze().cpu()

            predicts.append(predict)
        
        weights = np.ones(args.num_fold)
        weights = weights / np.sum(weights)

        predicts = np.average(predicts, axis=0, weights=weights)

        result_df = pl.DataFrame(
            data=[[series_uid] + predicts.tolist()],
            schema=[ID_COL, *LABEL_COLS],
            orient='row'
        )
    except Exception as e:
        # Return a fallback dataframe with the correct schema
        result_df = pl.DataFrame(
            data=[[series_uid] + [0.1] * len(LABEL_COLS)],
            schema=[ID_COL, *LABEL_COLS],
            orient='row'
        )
    finally:
        # This code is required to prevent "out of disk space" and "directory not empty" errors.
        # It deletes the shared folder and then immediately recreates it, ensuring it's
        # empty and ready for the next prediction.

        # shared_dir = '/kaggle/shared'
        shared_dir = './shared'
        shutil.rmtree(shared_dir, ignore_errors=True)
        os.makedirs(shared_dir, exist_ok=True)
    
    return result_df.drop(ID_COL)

In [11]:
results = predict('F:/ml_data_resource/kaggle/intracranial_aneurysm_detection/series_debug/1.2.826.0.1.3680043.8.498.10102361048562788202568222767625052953')
results.head()

Left Infraclinoid Internal Carotid Artery,Right Infraclinoid Internal Carotid Artery,Left Supraclinoid Internal Carotid Artery,Right Supraclinoid Internal Carotid Artery,Left Middle Cerebral Artery,Right Middle Cerebral Artery,Anterior Communicating Artery,Left Anterior Cerebral Artery,Right Anterior Cerebral Artery,Left Posterior Communicating Artery,Right Posterior Communicating Artery,Basilar Tip,Other Posterior Circulation,Aneurysm Present
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0.033347,0.07023,0.085212,0.090278,0.064246,0.069397,0.091479,0.020837,0.025355,0.018824,0.028396,0.023404,0.069278,0.426984
